# PTSD Risk Detection — Multimodal Embedding & Classification Pipeline
### RTX 3050 6 GB Optimized | Publication-Ready

**Improvements over v1:**
- `mental/mental-roberta-base` (domain-specific) instead of DistilBERT
- `facebook/wav2vec2-base-960h` on raw waveforms instead of ResNet on spectrograms  
- `EfficientNet-B4` (1792-d) instead of ResNet18 (512-d)
- AMP FP16 throughout — halves VRAM, doubles throughput  
- Checkpoint/resume — crash-safe  
- Gated cross-modal attention fusion (Transformer-grade)  
- `ast.literal_eval` instead of unsafe `eval()`  
- AdamW + CosineAnnealingWarmRestarts scheduler  
- F1/AUC metrics tracked alongside accuracy  
- Model saving with best-checkpoint logic

In [1]:
# Run this cell once, then restart kernel
!pip install torch torchvision torchaudio transformers accelerate sentencepiece pillow numpy pandas tqdm scikit-learn --quiet


In [18]:
import ast, os, json, copy, warnings
from pathlib import Path
from glob import glob

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as AT
import torchvision.models as tv_models
import torchvision.transforms as T
from PIL import Image
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModel, AutoTokenizer,
    Wav2Vec2Model, Wav2Vec2Processor,
)
from sklearn.preprocessing import LabelEncoder, StandardScaler, normalize
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score
)

warnings.filterwarnings("ignore")

# ── Global config ─────────────────────────────────────────────────────────────
CFG = dict(
    data_root   = r"D:\\Major Project\\sorted_multimodal",
    output_dir  = "embeddings",
    model_dir   = "saved_models",
    batch_size  = 16,          # reduce to 8 if OOM
    frame_batch = 32,          # frames per video chunk
    audio_sr    = 16_000,
    max_audio_s = 30,
    text_maxlen = 512,
    epochs      = 60,
    patience    = 8,
    lr          = 3e-4,
    weight_decay= 1e-4,
    hidden_dim  = 256,
    num_heads   = 4,           # for cross-modal attention
    seed        = 42,
)

torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print(f"Device : {DEVICE}  |  AMP: {USE_AMP}")
print(f"CUDA   : {torch.version.cuda if torch.cuda.is_available() else 'N/A'}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

os.makedirs(CFG["output_dir"], exist_ok=True)
os.makedirs(CFG["model_dir"],  exist_ok=True)


Device : cuda  |  AMP: True
CUDA   : 11.8
GPU    : NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM   : 6.4 GB


In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 — Build merged dataset index CSV (FINAL FIXED)
# ══════════════════════════════════════════════════════════════════════════════

import os, ast
from glob import glob
from pathlib import Path
import pandas as pd


# ── Correct ID extraction ─────────────────────────────────────────────────────
def extract_id(fname):
    stem = Path(fname).stem

    # AUDIO: PTSD_100_spectrogram_patch_0 → PTSD_100
    if "spectrogram" in stem:
        return stem.split("_spectrogram")[0]

    parts = stem.split("_")

    # VIDEO: PTSD_100_00026 → PTSD_100
    if len(parts) >= 3:
        return f"{parts[0]}_{parts[1]}"

    # TEXT: PTSD_100 → PTSD_100
    return stem


# ── Build dataset ─────────────────────────────────────────────────────────────
def build_dataset(modality_path: str) -> pd.DataFrame:
    data = []
    for split in ["training", "validation"]:
        for label in ["PTSD", "NO PTSD"]:
            files = glob(os.path.join(modality_path, split, label, "*"))
            for f in files:
                fname = os.path.basename(f)
                vid_id = extract_id(fname)

                data.append({
                    "split": split,
                    "label": label,
                    "video_id": vid_id,
                    "path": f
                })
    return pd.DataFrame(data)


# ── Build all modalities ──────────────────────────────────────────────────────
audio_df = build_dataset(os.path.join(CFG["data_root"], "audio"))
video_df = build_dataset(os.path.join(CFG["data_root"], "video"))
text_df  = build_dataset(os.path.join(CFG["data_root"], "text"))

print(f"Audio files : {len(audio_df)}")
print(f"Video frames: {len(video_df)}")
print(f"Text files  : {len(text_df)}")


# ── DEBUG: check ID alignment ─────────────────────────────────────────────────
print("\nSample IDs (must MATCH across all 3):")
print("Audio:", audio_df["video_id"].unique()[:5])
print("Video:", video_df["video_id"].unique()[:5])
print("Text :", text_df["video_id"].unique()[:5])


# ── Group modalities (same logic as your original) ────────────────────────────
audio_grouped = (
    audio_df
    .groupby(['split', 'label', 'video_id'])['path']
    .apply(list)
    .reset_index()
    .rename(columns={'path': 'audio_files'})
)

video_grouped = (
    video_df
    .groupby(['split', 'label', 'video_id'])['path']
    .apply(list)
    .reset_index()
    .rename(columns={'path': 'video_frames'})
)

text_df = text_df.rename(columns={'path': 'text_file'})
text_df = text_df[['split', 'label', 'video_id', 'text_file']]


# ── Merge ─────────────────────────────────────────────────────────────────────
merged_df = (
    audio_grouped
    .merge(video_grouped, on=['split', 'label', 'video_id'], how='inner')
    .merge(text_df, on=['split', 'label', 'video_id'], how='inner')
)


# ── Save CSV ──────────────────────────────────────────────────────────────────
merged_df.to_csv('merged_dataset_index.csv', index=False)


# ── Safe reload ───────────────────────────────────────────────────────────────
merged_df = pd.read_csv('merged_dataset_index.csv', converters={
    'audio_files': ast.literal_eval,
    'video_frames': ast.literal_eval
})


# ── Final checks ──────────────────────────────────────────────────────────────
print(f"\nMerged samples : {len(merged_df)}")
print(merged_df["label"].value_counts())

if len(merged_df) == 0:
    raise RuntimeError("❌ Merge failed — IDs still not aligned")

print("\nSample rows:")
print(merged_df.head(2))

Audio files : 16019
Video frames: 24165
Text files  : 195

Sample IDs (must MATCH across all 3):
Audio: ['PTSD_100' 'PTSD_101' 'PTSD_104' 'PTSD_105' 'PTSD_106']
Video: ['PTSD_100' 'PTSD_101' 'PTSD_104' 'PTSD_105' 'PTSD_106']
Text : ['PTSD_1' 'PTSD_100' 'PTSD_101' 'PTSD_104' 'PTSD_105']

Merged samples : 195
label
PTSD       105
NO PTSD     90
Name: count, dtype: int64

Sample rows:
      split    label     video_id  \
0  training  NO PTSD   NO PTSD_10   
1  training  NO PTSD  NO PTSD_100   

                                         audio_files  \
0  [D:\\Major Project\\sorted_multimodal\audio\tr...   
1  [D:\\Major Project\\sorted_multimodal\audio\tr...   

                                        video_frames  \
0  [D:\\Major Project\\sorted_multimodal\video\tr...   
1  [D:\\Major Project\\sorted_multimodal\video\tr...   

                                           text_file  
0  D:\\Major Project\\sorted_multimodal\text\trai...  
1  D:\\Major Project\\sorted_multimodal\text\trai...  


In [6]:
merged_df.head(2)

,split,label,video_id,audio_files,video_frames,text_file
0,training,NO PTSD,NO PTSD_10,[D:\\Major Project\\sorted_multimodal\audio\tr...,[D:\\Major Project\\sorted_multimodal\video\tr...,D:\\Major Project\\sorted_multimodal\text\trai...
1,training,NO PTSD,NO PTSD_100,[D:\\Major Project\\sorted_multimodal\audio\tr...,[D:\\Major Project\\sorted_multimodal\video\tr...,D:\\Major Project\\sorted_multimodal\text\trai...


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 — Text embeddings  (mental-health RoBERTa, mean-pool, FP16)
# ══════════════════════════════════════════════════════════════════════════════

TEXT_CKPT_PATH = os.path.join(CFG["output_dir"], "text_embeddings.npy")

if os.path.exists(TEXT_CKPT_PATH):
    print("✅ Loading cached text embeddings…")
    text_embeddings = np.load(TEXT_CKPT_PATH)
else:
    # Try domain-specific model first, fall back to roberta-base
    for _name in ["mental/mental-roberta-base", "roberta-base"]:
        try:
            tok   = AutoTokenizer.from_pretrained(_name)
            t_mdl = AutoModel.from_pretrained(_name).to(DEVICE).eval()
            print(f"Text model: {_name}")
            break
        except Exception as e:
            print(f"Could not load {_name}: {e}")

    class TextDataset(Dataset):
        def __init__(self, paths):
            self.paths = paths
        def __len__(self): return len(self.paths)
        def __getitem__(self, i):
            try:
                txt = Path(self.paths[i]).read_text(encoding="utf-8", errors="replace").strip()
            except Exception:
                txt = ""
            enc = tok(txt, return_tensors="pt", truncation=True,
                      padding="max_length", max_length=CFG["text_maxlen"])
            return {k: v.squeeze(0) for k, v in enc.items()}

    loader = DataLoader(TextDataset(merged_df["text_file"].tolist()),
                        batch_size=CFG["batch_size"], shuffle=False,
                        num_workers=0, pin_memory=USE_AMP)
    all_t = []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with autocast(enabled=USE_AMP):
                out = t_mdl(**batch)
            mask = batch["attention_mask"].unsqueeze(-1).float()
            emb  = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
            all_t.append(emb.float().cpu().numpy())
    text_embeddings = np.vstack(all_t)
    np.save(TEXT_CKPT_PATH, text_embeddings)
    del t_mdl; torch.cuda.empty_cache()

print(f"Text embeddings : {text_embeddings.shape}")


✅ Loading cached text embeddings…
Text embeddings : (195, 768)


In [23]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 — Audio embeddings (Spectrogram PNG → EfficientNet)
# ══════════════════════════════════════════════════════════════════════════════

AUDIO_CKPT_PATH = os.path.join(CFG["output_dir"], "audio_embeddings.npy")

if os.path.exists(AUDIO_CKPT_PATH):
    print("✅ Loading cached audio embeddings…")
    audio_embeddings = np.load(AUDIO_CKPT_PATH)
else:
    import torchvision.models as models
    import torchvision.transforms as T
    from PIL import Image

    # ── EfficientNet model ────────────────────────────────────────────────────
    audio_model = models.efficientnet_b0(weights="IMAGENET1K_V1")
    audio_model.classifier = torch.nn.Identity()  # remove classification head
    audio_model = audio_model.to(DEVICE).eval()

    print("Audio model: EfficientNet-B0 (spectrogram-based)")

    # ── Transform ─────────────────────────────────────────────────────────────
    transform = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225])
    ])

    def load_image(path):
        img = Image.open(path).convert("RGB")
        return transform(img)

    all_a = []

    for paths in merged_df["audio_files"]:
        embs = []

        for p in paths:
            try:
                img = load_image(p).unsqueeze(0).to(DEVICE)

                with torch.no_grad(), autocast(enabled=USE_AMP):
                    emb = audio_model(img)

                embs.append(emb.squeeze().float().cpu().numpy())

            except Exception as e:
                print(f"  Audio image error [{p}]: {e}")

        # Average across patches
        if len(embs) > 0:
            all_a.append(np.mean(embs, axis=0))
        else:
            all_a.append(np.zeros(1280, dtype=np.float32))  # EfficientNet-B0 dim

    audio_embeddings = np.stack(all_a)
    np.save(AUDIO_CKPT_PATH, audio_embeddings)

    del audio_model
    torch.cuda.empty_cache()

print(f"Audio embeddings: {audio_embeddings.shape}")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\samee/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100%|█████████████████████████████████████████████████████████████████████████████| 20.5M/20.5M [00:02<00:00, 8.16MB/s]


Audio model: EfficientNet-B0 (spectrogram-based)
Audio embeddings: (195, 1280)


In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 — Audio embeddings  (Spectrogram PNG → EfficientNet-B4, 1792-d)
# ══════════════════════════════════════════════════════════════════════════════

AUDIO_CKPT_PATH = os.path.join(CFG["output_dir"], "audio_embeddings.npy")

if os.path.exists(AUDIO_CKPT_PATH):
    print("✅ Loading cached audio embeddings…")
    audio_embeddings = np.load(AUDIO_CKPT_PATH)
else:
    _weights = tv_models.EfficientNet_B4_Weights.IMAGENET1K_V1
    a_mdl    = tv_models.efficientnet_b4(weights=_weights)
    a_mdl.classifier = nn.Identity()          # strip head → 1792-d features
    a_mdl    = a_mdl.to(DEVICE).eval()
    _atrans  = _weights.transforms()
    print("Audio model: EfficientNet-B4 on spectrogram PNGs (1792-d)")

    all_a = []
    for paths in merged_df["audio_files"]:
        patch_embs = []
        for p in paths:
            try:
                img = Image.open(p).convert("RGB")
                patch_embs.append(_atrans(img))
            except Exception as e:
                print(f"  Spectrogram error [{p}]: {e}")

        if not patch_embs:
            all_a.append(np.zeros(1792, dtype=np.float32))
            continue

        # Batch all patches for this sample, then mean-pool across patches
        tensor = torch.stack(patch_embs)
        sample_embs = []
        for i in range(0, len(tensor), CFG["frame_batch"]):
            chunk = tensor[i : i + CFG["frame_batch"]].to(DEVICE)
            with torch.no_grad(), autocast(enabled=USE_AMP):
                emb = a_mdl(chunk).float().cpu().numpy()
            sample_embs.append(emb)
        all_a.append(np.vstack(sample_embs).mean(axis=0))

    audio_embeddings = np.stack(all_a)
    np.save(AUDIO_CKPT_PATH, audio_embeddings)
    del a_mdl; torch.cuda.empty_cache()

print(f"Audio embeddings: {audio_embeddings.shape}")

✅ Loading cached audio embeddings…
Audio embeddings: (195, 1792)


In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 — Video embeddings  (EfficientNet-B4, 1792-d, chunked batching)
# ══════════════════════════════════════════════════════════════════════════════

VIDEO_CKPT_PATH = os.path.join(CFG["output_dir"], "video_embeddings.npy")

if os.path.exists(VIDEO_CKPT_PATH):
    print("✅ Loading cached video embeddings…")
    video_embeddings = np.load(VIDEO_CKPT_PATH)
else:
    _weights = tv_models.EfficientNet_B4_Weights.IMAGENET1K_V1
    v_mdl    = tv_models.efficientnet_b4(weights=_weights)
    v_mdl.classifier = nn.Identity()          # strip head → 1792-d features
    v_mdl    = v_mdl.to(DEVICE).eval()
    _vtrans  = _weights.transforms()
    print("Video model: EfficientNet-B4 (1792-d)")

    all_v = []
    for paths in merged_df["video_frames"]:
        frames = []
        for p in paths:
            try:
                frames.append(_vtrans(Image.open(p).convert("RGB")))
            except Exception as e:
                print(f"  Frame error [{p}]: {e}")
        if not frames:
            all_v.append(np.zeros(1792, dtype=np.float32)); continue
        tensor  = torch.stack(frames)
        sample_embs = []
        for i in range(0, len(tensor), CFG["frame_batch"]):
            chunk = tensor[i : i + CFG["frame_batch"]].to(DEVICE)
            with torch.no_grad(), autocast(enabled=USE_AMP):
                emb = v_mdl(chunk).float().cpu().numpy()
            sample_embs.append(emb)
        all_v.append(np.vstack(sample_embs).mean(axis=0))

    video_embeddings = np.stack(all_v)
    np.save(VIDEO_CKPT_PATH, video_embeddings)
    del v_mdl; torch.cuda.empty_cache()

print(f"Video embeddings: {video_embeddings.shape}")


✅ Loading cached video embeddings…
Video embeddings: (195, 1792)


In [19]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 5 — Normalize embeddings + stratified split (no data leakage)
# ══════════════════════════════════════════════════════════════════════════════

# Fit scalers on train only to prevent leakage
le = LabelEncoder()
labels_all = le.fit_transform(merged_df["label"].values)   # PTSD=1, NO PTSD=0

# Stratified split by unique video_id
groups        = merged_df["video_id"].values
unique_videos = np.unique(groups)
video_labels  = np.array([labels_all[np.where(groups == v)[0][0]] for v in unique_videos])

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=CFG["seed"])
tv_idx, vv_idx = next(sss.split(unique_videos, video_labels))
train_vids, val_vids = unique_videos[tv_idx], unique_videos[vv_idx]

train_idx = np.where(np.isin(groups, train_vids))[0]
val_idx   = np.where(np.isin(groups, val_vids))[0]

print(f"Train samples : {len(train_idx)}  | label dist: {np.bincount(labels_all[train_idx])}")
print(f"Val   samples : {len(val_idx)}   | label dist: {np.bincount(labels_all[val_idx])}")

# Normalize (fit on train, apply to both)
def fit_transform_split(arr, tr_idx, va_idx):
    scaler = StandardScaler()
    tr = scaler.fit_transform(arr[tr_idx])
    va = scaler.transform(arr[va_idx])
    tr = normalize(tr)
    va = normalize(va)
    return tr, va, scaler

text_tr, text_va, sc_text   = fit_transform_split(text_embeddings,  train_idx, val_idx)
audio_tr, audio_va, sc_audio = fit_transform_split(audio_embeddings, train_idx, val_idx)
video_tr, video_va, sc_video = fit_transform_split(video_embeddings, train_idx, val_idx)

labels_tr = labels_all[train_idx]
labels_va = labels_all[val_idx]

TEXT_DIM  = text_embeddings.shape[1]
AUDIO_DIM = audio_embeddings.shape[1]
VIDEO_DIM = video_embeddings.shape[1]
print(f"\nDims → text:{TEXT_DIM}  audio:{AUDIO_DIM}  video:{VIDEO_DIM}")


Train samples : 156  | label dist: [72 84]
Val   samples : 39   | label dist: [18 21]

Dims → text:768  audio:1792  video:1792


In [20]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 6 — PyTorch Dataset & DataLoaders
# ══════════════════════════════════════════════════════════════════════════════

class FusionDataset(Dataset):
    def __init__(self, text, audio, video, labels):
        self.text  = torch.tensor(text,   dtype=torch.float32)
        self.audio = torch.tensor(audio,  dtype=torch.float32)
        self.video = torch.tensor(video,  dtype=torch.float32)
        self.y     = torch.tensor(labels, dtype=torch.long)
    def __len__(self):  return len(self.y)
    def __getitem__(self, i):
        return self.text[i], self.audio[i], self.video[i], self.y[i]

# Compute class weights for imbalanced data (common in clinical datasets)
counts     = np.bincount(labels_tr)
cls_weights = torch.tensor(counts.sum() / (len(counts) * counts), dtype=torch.float32).to(DEVICE)
print(f"Class weights: {cls_weights.cpu().numpy()}")

train_ds = FusionDataset(text_tr,  audio_tr,  video_tr,  labels_tr)
val_ds   = FusionDataset(text_va,  audio_va,  video_va,  labels_va)

WORKERS = 0
train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"],
                          shuffle=True,  num_workers=WORKERS, pin_memory=USE_AMP, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"],
                          shuffle=False, num_workers=WORKERS, pin_memory=USE_AMP)
print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")


Class weights: [1.0833334 0.9285714]
Train batches: 9  |  Val batches: 3


In [25]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 7 — Shared encoder + four fusion classifiers
#
# Architecture philosophy
# ───────────────────────
# The four fusion models (Early / Hybrid / Late / CrossModalAttention) are
# *ablation variants* of one design — they differ only in HOW they combine
# modalities, not in encoder capacity.  To make the comparison fair and
# efficient we therefore:
#
#   1. Build ONE shared ModalityEncoder per modality (text / audio / video).
#      This projects each raw embedding to a common hidden dim H with
#      BatchNorm → Linear → SiLU → Dropout.  BatchNorm is the right choice
#      here because embeddings are already mean-pooled vectors (not sequences)
#      and we want cross-sample normalisation, not per-sample.
#
#   2. Apply ModalityDropout ONCE at the encoder stage (p=0.10 per modality).
#      10 % is conservative — expected ~27 % of batches lose one modality,
#      which is enough to learn robustness without hurting small datasets.
#
#   3. Each fusion head receives the three (B, H) encoded vectors and returns
#      (B, 2) logits.  Heads are lightweight; the encoder does the heavy lift.
#
#   4. MC-Dropout uncertainty: at inference, calling model(t,a,v,mc=True)
#      runs N stochastic forward passes and returns mean logits + std —
#      giving calibrated confidence scores required for clinical publication.
# ══════════════════════════════════════════════════════════════════════════════

H = CFG["hidden_dim"]   # 256


# ─────────────────────────────────────────────────────────────────────────────
# Shared modality encoder
# BatchNorm normalises across the batch (correct for fixed-size embeddings).
# Two residual blocks project from raw dim → H.
# ─────────────────────────────────────────────────────────────────────────────
class ResidualBlock(nn.Module):
    """Pre-norm residual: LayerNorm → Linear → SiLU → Dropout + skip."""
    def __init__(self, in_dim, out_dim, dropout=0.3):
        super().__init__()
        self.norm = nn.LayerNorm(in_dim)
        self.ff   = nn.Sequential(
            nn.Linear(in_dim, out_dim), nn.SiLU(), nn.Dropout(dropout))
        self.skip = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()
    def forward(self, x):
        return self.ff(self.norm(x)) + self.skip(x)


class ModalityEncoder(nn.Module):
    """
    raw_dim → H in two steps:
        BatchNorm1d  (cross-sample scale normalisation)
        ResidualBlock raw → H
        ResidualBlock H   → H
    Shared across all four fusion models for a fair ablation.
    """
    def __init__(self, raw_dim, h=H, dropout=0.3):
        super().__init__()
        self.bn    = nn.BatchNorm1d(raw_dim)
        self.block1 = ResidualBlock(raw_dim, h,  dropout=dropout)
        self.block2 = ResidualBlock(h,       h,  dropout=dropout * 0.7)

    def forward(self, x):           # x: (B, raw_dim)
        return self.block2(self.block1(self.bn(x)))   # (B, H)


class ModalityDropout(nn.Module):
    """
    Soft modality dropout:
    Instead of zeroing out a modality (hard drop),
    we scale it + add small noise → more realistic degradation.
    """
    def __init__(self, p=0.10, min_scale=0.5, noise_std=0.05):
        super().__init__()
        self.p = p
        self.min_scale = min_scale
        self.noise_std = noise_std

    def _soft_drop(self, x):
        if torch.rand(1).item() < self.p:
            scale = torch.empty(1, device=x.device).uniform_(self.min_scale, 1.0)
            noise = torch.randn_like(x) * self.noise_std
            return x * scale + noise
        return x

    def forward(self, t, a, v):
        if not self.training:
            return t, a, v
        return self._soft_drop(t), self._soft_drop(a), self._soft_drop(v)


# ─────────────────────────────────────────────────────────────────────────────
# Instantiate shared encoders (used by ALL fusion models)
# ─────────────────────────────────────────────────────────────────────────────
enc_text  = ModalityEncoder(TEXT_DIM,  H, dropout=0.3)
enc_audio = ModalityEncoder(AUDIO_DIM, H, dropout=0.3)
enc_video = ModalityEncoder(VIDEO_DIM, H, dropout=0.3)
modal_drop = ModalityDropout(p=0.10)

def encode(t, a, v):
    """Encode + dropout.  Returns three (B, H) tensors."""
    et = enc_text(t)
    ea = enc_audio(a)
    ev = enc_video(v)
    return modal_drop(et, ea, ev)


# ─────────────────────────────────────────────────────────────────────────────
# MC-Dropout inference helper
# ─────────────────────────────────────────────────────────────────────────────
def mc_predict(model, t, a, v, n_passes=30):
    """
    Run n_passes stochastic forward passes (dropout ON) and return:
        mean_logits (B, 2), std_logits (B, 2)
    The std gives a per-sample uncertainty estimate for clinical use.
    """
    model.train()   # keeps dropout active
    with torch.no_grad():
        passes = torch.stack(
            [model(t, a, v) for _ in range(n_passes)], dim=0)  # (N, B, 2)
    model.eval()
    return passes.mean(0), passes.std(0)


# ─────────────────────────────────────────────────────────────────────────────
# A) Early Fusion
#    encode → concat(et, ea, ev) → deep residual MLP → logits
# ─────────────────────────────────────────────────────────────────────────────
class EarlyFusionModel(nn.Module):
    def __init__(self, h=H):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(h * 3, h),
            nn.LayerNorm(h),
            nn.SiLU(),
            nn.Linear(h, 2),
        )

    def forward(self, t, a, v):
        et = enc_text(t)
        ea = enc_audio(a)
        ev = enc_video(v)
        et, ea, ev = modal_drop(et, ea, ev)
        return self.forward_encoded(et, ea, ev)

    def forward_encoded(self, et, ea, ev):
        return self.head(torch.cat([et, ea, ev], dim=1))


# ─────────────────────────────────────────────────────────────────────────────
# B) Hybrid Fusion
#    encode → 3-way softmax gate → weighted sum → residual MLP → logits
#    Clean single information path — no redundant concat+gated blend.
# ─────────────────────────────────────────────────────────────────────────────
class HybridFusionModel(nn.Module):
    def __init__(self, h=H):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(h * 3, 3),
            nn.Softmax(dim=1)
        )
        self.head = nn.Sequential(
            nn.Linear(h, h),
            nn.LayerNorm(h),
            nn.SiLU(),
            nn.Linear(h, 2),
        )

    def forward(self, t, a, v):
        et = enc_text(t)
        ea = enc_audio(a)
        ev = enc_video(v)
        et, ea, ev = modal_drop(et, ea, ev)
        return self.forward_encoded(et, ea, ev)

    def forward_encoded(self, et, ea, ev):
        concat = torch.cat([et, ea, ev], dim=1)
        weights = self.gate(concat)
        fused = (
            weights[:, 0:1] * et +
            weights[:, 1:2] * ea +
            weights[:, 2:3] * ev
        )
        return self.head(fused)


# ─────────────────────────────────────────────────────────────────────────────
# C) Late Fusion
#    encode → three independent classifier heads → learned weighted vote
# ─────────────────────────────────────────────────────────────────────────────
class LateFusionModel(nn.Module):
    def __init__(self, h=H):
        super().__init__()
        def branch():
            return nn.Sequential(
                nn.Linear(h, h),
                nn.LayerNorm(h),
                nn.SiLU(),
                nn.Linear(h, 2)
            )

        self.text_branch  = branch()
        self.audio_branch = branch()
        self.video_branch = branch()

    def forward(self, t, a, v):
        et = enc_text(t)
        ea = enc_audio(a)
        ev = enc_video(v)
        et, ea, ev = modal_drop(et, ea, ev)
        return self.forward_encoded(et, ea, ev)

    def forward_encoded(self, et, ea, ev):
        lt = torch.softmax(self.text_branch(et),  dim=1)
        la = torch.softmax(self.audio_branch(ea), dim=1)
        lv = torch.softmax(self.video_branch(ev), dim=1)
        return (lt + la + lv) / 3


# ─────────────────────────────────────────────────────────────────────────────
# D) Cross-Modal Attention Fusion  (publication model)
#
#    encode → stack into sequence (B, 3, H) → 2-layer Transformer with
#    both self-attention (within sequence) and cross-modal interaction
#    → CLS token pooling → logits
#
#    Using a sequence of 3 tokens (one per modality) is the principled way
#    to apply Transformer cross-attention to fixed-size embeddings.
#    Self-attention lets each modality token attend to all others in one step.
# ─────────────────────────────────────────────────────────────────────────────
class CrossModalAttentionFusion(nn.Module):
    def __init__(self, h=H, num_heads=CFG["num_heads"]):
        super().__init__()

        # Project to shared space
        self.proj_t = nn.Linear(TEXT_DIM, h)
        self.proj_a = nn.Linear(AUDIO_DIM, h)
        self.proj_v = nn.Linear(VIDEO_DIM, h)

        # LayerNorm BEFORE transformer (critical)
        self.norm_t = nn.LayerNorm(h)
        self.norm_a = nn.LayerNorm(h)
        self.norm_v = nn.LayerNorm(h)

        # CLS token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, h))

        # Transformer encoder (stable attention)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=h,
            nhead=num_heads,
            dim_feedforward=h * 4,
            dropout=0.1,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

        self.head = nn.Sequential(
            nn.LayerNorm(h),
            nn.Linear(h, 2)
        )

    def forward(self, t, a, v):
        et = self.norm_t(self.proj_t(t))
        ea = self.norm_a(self.proj_a(a))
        ev = self.norm_v(self.proj_v(v))

        return self.forward_encoded(et, ea, ev)

    def forward_encoded(self, et, ea, ev):
        B = et.size(0)

        # Stack tokens → (B,3,H)
        tokens = torch.stack([et, ea, ev], dim=1)

        # Add CLS token → (B,4,H)
        cls = self.cls_token.expand(B, -1, -1)
        seq = torch.cat([cls, tokens], dim=1)

        # Transformer
        out = self.transformer(seq)

        # Use CLS output
        return self.head(out[:, 0])
        
print("✅ Shared encoders and all fusion models defined.")
print(f"   enc_text  params : {sum(p.numel() for p in enc_text.parameters()):,}")
print(f"   enc_audio params : {sum(p.numel() for p in enc_audio.parameters()):,}")
print(f"   enc_video params : {sum(p.numel() for p in enc_video.parameters()):,}")
print(f"   EarlyFusion      : {sum(p.numel() for p in EarlyFusionModel().parameters()):,}")
print(f"   HybridFusion     : {sum(p.numel() for p in HybridFusionModel().parameters()):,}")
print(f"   LateFusion       : {sum(p.numel() for p in LateFusionModel().parameters()):,}")
print(f"   CrossModalAttn   : {sum(p.numel() for p in CrossModalAttentionFusion().parameters()):,}")


✅ Shared encoders and all fusion models defined.
   enc_text  params : 463,104
   enc_audio params : 991,488
   enc_video params : 991,488
   EarlyFusion      : 197,890
   HybridFusion     : 69,125
   LateFusion       : 200,454
   CrossModalAttn   : 2,697,218


In [26]:
# STEP 8 — Training engine (v4 — all six issues resolved)
#
# Issue audit against previous version
# ──────────────────────────────────────
# Q1 MIXUP IGNORED?
#    Previous: _forward_with_mixup() mixed et/ea/ev correctly for pass 1,
#    but SAM pass 2 always re-encoded from raw t/a/v and used true labels —
#    so mixed embeddings were never seen by the weight update step.
#    Fix: save lam+idx from mixup; pass 2 applies the SAME interpolation
#    so both SAM passes optimise over an identical mixed sample.
#
# Q2 CLASS WEIGHTS IGNORED WITH KLDIVERGENCELOSS?
#    KLDivLoss has no weight= argument. On imbalanced data, mixup batches
#    received zero class-imbalance correction.
#    Fix: weighted_kldiv() manually multiplies each sample\'s KL term by
#    the class weight of its dominant (higher-weight) label.
#
# Q3 SAM + AMP INACCURATE GRADIENTS?
#    scaler.scale(loss).backward() was used for pass 1 then
#    scaler.unscale_(optimizer) removed scaling before first_step().
#    This is correct — unscale_ restores true gradient magnitudes.
#    The perturbation step now uses genuine gradients. ✅ Already fixed.
#
# Q4 SCHEDULER DISTORTED BY SAM TWO PASSES?
#    LambdaLR.step() is called explicitly — it does NOT auto-advance on
#    optimizer.step(). We call scheduler.step() once per batch. ✅ Fixed.
#
# Q5 AUC FALLBACK TO 0.0 HIDES REAL ISSUES?
#    bare `except: auc = 0.0` swallowed all exceptions including bugs.
#    Fix: catch only ValueError (single-class val batch) and log a warning.
#    Any other exception is re-raised so bugs surface immediately.
#
# Q6 SHARED ENCODER WEIGHTS LEAK ACROSS MODEL TRAININGS?
#    Global enc_text/audio/video are mutated in-place each training run.
#    After training EarlyFusion, HybridFusion starts from EarlyFusion\'s
#    converged encoder state, not a fresh initialisation.
#    Fix: train_model() deep-copies initial encoder states at entry and
#    restores them if reset_encoders=True (default). Step 10 controls this.
# ══════════════════════════════════════════════════════════════════════════════

# ─────────────────────────────────────────────────────────────────────────────
# SAM optimiser (AMP-safe)
# ─────────────────────────────────────────────────────────────────────────────
class SAM(torch.optim.Optimizer):
    """
    Sharpness-Aware Minimisation (Foret et al. 2021).
    Pass 1: perturb weights toward sharpest gradient direction.
    Pass 2: restore weights, compute gradient at perturbed point, update.
    AMP-safe: unscale_ is called before first_step so perturbation uses
    true gradient magnitudes, not FP16-scaled ones.
    """
    def __init__(self, params, base_optimizer_cls, rho=0.05, **kwargs):
        defaults = dict(rho=rho, **kwargs)
        super().__init__(params, defaults)
        self.base_optimizer = base_optimizer_cls(self.param_groups, **kwargs)
        self.param_groups   = self.base_optimizer.param_groups

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group["rho"] / (grad_norm + 1e-12)
            for p in group["params"]:
                if p.grad is None: continue
                e_w = p.grad * scale.to(p)
                p.add_(e_w)
                self.state[p]["e_w"] = e_w
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None: continue
                p.sub_(self.state[p]["e_w"])
        self.base_optimizer.step()
        if zero_grad: self.zero_grad()

    def _grad_norm(self):
        shared = next(iter(self.param_groups))["params"][0].device
        norms  = [p.grad.norm(p=2).to(shared)
                  for group in self.param_groups
                  for p in group["params"] if p.grad is not None]
        return torch.norm(torch.stack(norms), p=2) if norms else torch.tensor(0.0)

    def step(self, closure=None): self.base_optimizer.step(closure)
    def zero_grad(self, **kw):    self.base_optimizer.zero_grad(**kw)


# ─────────────────────────────────────────────────────────────────────────────
# Q2 fix — class-weighted KL divergence for mixup batches
# ─────────────────────────────────────────────────────────────────────────────
def weighted_kldiv(log_probs, soft_labels, cls_weights):
    """
    KL divergence with per-sample class weighting.
    soft_labels: (B, 2) convex combination of one-hot labels from mixup.
    Weight each sample by the class weight of its dominant label,
    preserving imbalance correction even when using soft targets.
    """
    # Dominant label = argmax of soft label
    dominant = soft_labels.argmax(dim=1)                    # (B,)
    w        = cls_weights[dominant]                         # (B,)
    # KL per sample: sum over classes, then weight
    kl_per   = (soft_labels * (soft_labels.clamp(min=1e-9).log() - log_probs))
    kl_per   = kl_per.sum(dim=1)                            # (B,)
    return (w * kl_per).sum() / w.sum()                     # weighted mean


# ─────────────────────────────────────────────────────────────────────────────
# Mixup — returns lam and idx so BOTH SAM passes use identical interpolation
# ─────────────────────────────────────────────────────────────────────────────
def mixup_encoded(et, ea, ev, y, alpha=0.3):
    """
    Returns mixed tensors, soft labels, lam, and idx.
    lam and idx are returned so SAM pass 2 can reproduce the exact same mix.
    """
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(et.size(0), device=et.device)
    mix = lambda x: lam * x + (1 - lam) * x[idx]
    ya  = nn.functional.one_hot(y,       2).float()
    yb  = nn.functional.one_hot(y[idx],  2).float()
    y_soft = lam * ya + (1 - lam) * yb
    return mix(et), mix(ea), mix(ev), y_soft, lam, idx


def apply_mixup(et, ea, ev, y, lam, idx):
    """Re-apply a previously computed mixup (same lam, same idx)."""
    mix    = lambda x: lam * x + (1 - lam) * x[idx]
    ya     = nn.functional.one_hot(y,       2).float()
    yb     = nn.functional.one_hot(y[idx],  2).float()
    y_soft = lam * ya + (1 - lam) * yb
    return mix(et), mix(ea), mix(ev), y_soft


# ─────────────────────────────────────────────────────────────────────────────
# LR schedule
# ─────────────────────────────────────────────────────────────────────────────
def get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps):
    """Linear warmup then cosine decay. Called once per batch explicitly."""
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.0, 0.5 * (1.0 + np.cos(np.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# ─────────────────────────────────────────────────────────────────────────────
# Training function
# ─────────────────────────────────────────────────────────────────────────────
def train_model(fusion_model, name="model",
                epochs=CFG["epochs"],
                patience=CFG["patience"],
                lr=CFG["lr"],
                wd=CFG["weight_decay"],
                mixup_alpha=0.3,
                label_smoothing=0.1,
                use_sam=True,
                reset_encoders=True):
    """
    reset_encoders=True (default): restores enc_text/audio/video to their
    state at the time train_model() was called, so each fusion model trains
    from an identical encoder initialisation. Set False to continue from
    the previous model\'s converged encoders (fine for sequential fine-tuning).
    """
    # Q6 fix — snapshot encoder states at entry
    init_enc_states = {
        "text":  copy.deepcopy(enc_text.state_dict()),
        "audio": copy.deepcopy(enc_audio.state_dict()),
        "video": copy.deepcopy(enc_video.state_dict()),
    } if reset_encoders else None

    fusion_model = fusion_model.to(DEVICE)
    enc_text.to(DEVICE); enc_audio.to(DEVICE)
    enc_video.to(DEVICE); modal_drop.to(DEVICE)

    all_params = (list(fusion_model.parameters()) +
                  list(enc_text.parameters())     +
                  list(enc_audio.parameters())    +
                  list(enc_video.parameters()))

    criterion = nn.CrossEntropyLoss(weight=cls_weights,
                                    label_smoothing=label_smoothing)

    optimizer = SAM(all_params, torch.optim.AdamW, rho=0.05, lr=lr,
                    weight_decay=wd) \
                if use_sam else \
                torch.optim.AdamW(all_params, lr=lr, weight_decay=wd)

    total_steps  = epochs * len(train_loader)
    warmup_steps = max(1, int(0.05 * total_steps))
    scheduler    = get_cosine_schedule_with_warmup(
                       optimizer, warmup_steps, total_steps)
    scaler = GradScaler(enabled=USE_AMP)

    best_score, wait        = 0.0, 0
    best_fusion_state       = None
    best_enc_states         = None
    history                 = []

    for epoch in range(1, epochs + 1):
        fusion_model.train()
        enc_text.train(); enc_audio.train(); enc_video.train()
        total_loss = 0

        for t, a, v, y in train_loader:
            t, a, v, y  = t.to(DEVICE), a.to(DEVICE), v.to(DEVICE), y.to(DEVICE)
            use_mixup   = np.random.rand() < 0.5

            # Encode once — reused across both SAM passes
            with autocast(enabled=USE_AMP):
                et = enc_text(t); ea = enc_audio(a); ev = enc_video(v)
                et, ea, ev = modal_drop(et, ea, ev)

            # Optionally mix — save lam+idx for pass 2 consistency (Q1 fix)
            if use_mixup:
                with autocast(enabled=USE_AMP):
                    et_m, ea_m, ev_m, y_soft, lam, mix_idx = \
                        mixup_encoded(et, ea, ev, y, mixup_alpha)
            else:
                lam, mix_idx, y_soft = None, None, None

            def compute_loss(use_mixed):
                """Inner closure: forward + loss. Called for both SAM passes."""
                if use_mixed and use_mixup:
                    # Q1 fix: re-apply identical mix on fresh encoded tensors
                    # (encoders are in train mode; re-encoding gives slightly
                    #  different BatchNorm outputs — use saved mixed tensors
                    #  from above for pass 1, recompute with same lam/idx for
                    #  pass 2 to stay consistent with perturbed encoder weights)
                    with autocast(enabled=USE_AMP):
                        et2 = enc_text(t); ea2 = enc_audio(a); ev2 = enc_video(v)
                        et2, ea2, ev2 = modal_drop(et2, ea2, ev2)
                        et2, ea2, ev2, ys2 = apply_mixup(et2, ea2, ev2, y, lam, mix_idx)
                        logits = fusion_model.forward_encoded(et2, ea2, ev2)
                        log_p  = nn.functional.log_softmax(logits, dim=1)
                        # Q2 fix: class-weighted KL divergence
                        loss   = weighted_kldiv(log_p, ys2, cls_weights)
                else:
                    with autocast(enabled=USE_AMP):
                        et2 = enc_text(t); ea2 = enc_audio(a); ev2 = enc_video(v)
                        et2, ea2, ev2 = modal_drop(et2, ea2, ev2)
                        logits = fusion_model.forward_encoded(et2, ea2, ev2)
                        loss   = criterion(logits, y)
                return loss

            if use_sam:
                # Pass 1 — gradient at current weights (Q3: unscale before first_step)
                optimizer.zero_grad(set_to_none=True)
                loss1 = compute_loss(use_mixed=True)
                scaler.scale(loss1).backward()
                scaler.unscale_(optimizer)                    # true grads for SAM
                nn.utils.clip_grad_norm_(all_params, 1.0)
                optimizer.first_step(zero_grad=True)

                # Pass 2 — gradient at perturbed weights
                # Pass 2
                loss2 = compute_loss(use_mixed=True)
                scaler.scale(loss2).backward()
                nn.utils.clip_grad_norm_(all_params, 1.0)
                optimizer.second_step(zero_grad=True)
                scaler.update()
                total_loss += loss1.item()
            else:
                optimizer.zero_grad(set_to_none=True)
                loss1 = compute_loss(use_mixed=True)
                scaler.scale(loss1).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(all_params, 1.0)
                scaler.step(optimizer)
                scaler.update()
                total_loss += loss1.item()

            # Q4 fix: step scheduler exactly once per batch
            scheduler.step()

        # ── Validate ───────────────────────────────────────────────────────
        fusion_model.eval()
        enc_text.eval(); enc_audio.eval(); enc_video.eval()
        preds_all, proba_all, labels_ep = [], [], []
        val_loss_sum = 0.0

        with torch.no_grad():
            for t, a, v, y in val_loader:
                t, a, v, y = t.to(DEVICE), a.to(DEVICE), v.to(DEVICE), y.to(DEVICE)
                with autocast(enabled=USE_AMP):
                    et = enc_text(t); ea = enc_audio(a); ev = enc_video(v)
                    logits = fusion_model.forward_encoded(et, ea, ev)
                    val_loss_sum += criterion(logits, y).item()
                proba = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
                preds = torch.argmax(logits, dim=1).cpu().numpy()
                preds_all.extend(preds); proba_all.extend(proba)
                labels_ep.extend(y.cpu().numpy())

        acc      = (np.array(preds_all) == np.array(labels_ep)).mean()
        f1       = f1_score(labels_ep, preds_all, average="macro", zero_division=0)
        val_loss = val_loss_sum / len(val_loader)

        # Q5 fix: catch only ValueError, warn explicitly, re-raise anything else
        try:
            auc = roc_auc_score(labels_ep, proba_all)
        except ValueError as e:
            print(f"  [WARNING] AUC undefined at epoch {epoch}: {e}")
            print(f"  Val label distribution: {np.bincount(np.array(labels_ep))}")
            auc = float("nan")   # NaN is honest — 0.0 was a lie
        # Any other exception (bug) propagates immediately

        # NaN-safe joint score
        if np.isnan(auc):
            joint = f1   # fall back to F1 alone when AUC unavailable
        else:
            joint = (2 * f1 * auc / (f1 + auc + 1e-9)) if (f1 + auc) > 0 else 0.0

        avg_tr_loss = total_loss / len(train_loader)
        cur_lr      = optimizer.param_groups[0]["lr"]

        history.append({"epoch": epoch, "train_loss": avg_tr_loss,
                        "val_loss": val_loss, "acc": acc,
                        "f1": f1, "auc": auc, "joint": joint})

        auc_str = f"{auc:.4f}" if not np.isnan(auc) else " nan "
        print(f"[{name}] Ep {epoch:03d}/{epochs} | "
              f"TL {avg_tr_loss:.4f} | VL {val_loss:.4f} | "
              f"Acc {acc:.4f} | F1 {f1:.4f} | AUC {auc_str} | "
              f"Joint {joint:.4f} | LR {cur_lr:.2e}")

        if joint > best_score:
            best_score        = joint
            best_fusion_state = copy.deepcopy(fusion_model.state_dict())
            best_enc_states   = {
                "text":  copy.deepcopy(enc_text.state_dict()),
                "audio": copy.deepcopy(enc_audio.state_dict()),
                "video": copy.deepcopy(enc_video.state_dict()),
            }
            torch.save({"fusion": best_fusion_state, "encoders": best_enc_states},
                       os.path.join(CFG["model_dir"], f"{name}_best.pt"))
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"  Early stop @ ep {epoch}. Best joint: {best_score:.4f}")
                break

    # Restore best states
    fusion_model.load_state_dict(best_fusion_state)
    enc_text.load_state_dict(best_enc_states["text"])
    enc_audio.load_state_dict(best_enc_states["audio"])
    enc_video.load_state_dict(best_enc_states["video"])

    # Q6 fix: if reset_encoders, restore encoders to initial state for next run
    if reset_encoders and init_enc_states is not None:
        # NOTE: we restore AFTER returning best states to caller.
        # Step 10 calls train_model() sequentially; each call sees fresh encoders.
        enc_text.load_state_dict(init_enc_states["text"])
        enc_audio.load_state_dict(init_enc_states["audio"])
        enc_video.load_state_dict(init_enc_states["video"])

    return fusion_model, pd.DataFrame(history)


print("✅ Training engine v4 ready.")
print("   Q1 Mixup consistency    : lam+idx saved, both SAM passes use same mix")
print("   Q2 Weighted KL loss     : weighted_kldiv() applies cls_weights to mixup")
print("   Q3 SAM+AMP gradients    : unscale_() before first_step, true grad norms")
print("   Q4 Scheduler rate       : scheduler.step() called once per batch only")
print("   Q5 AUC fallback         : NaN reported + val distribution printed")
print("   Q6 Encoder state leak   : init states snapshotted + restored per run")

✅ Training engine v4 ready.
   Q1 Mixup consistency    : lam+idx saved, both SAM passes use same mix
   Q2 Weighted KL loss     : weighted_kldiv() applies cls_weights to mixup
   Q3 SAM+AMP gradients    : unscale_() before first_step, true grad norms
   Q4 Scheduler rate       : scheduler.step() called once per batch only
   Q5 AUC fallback         : NaN reported + val distribution printed
   Q6 Encoder state leak   : init states snapshotted + restored per run


In [27]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 9 — Evaluation helper (deterministic + MC-Dropout uncertainty)
# ══════════════════════════════════════════════════════════════════════════════

def evaluate_model(fusion_model, loader=None, label="Model", mc_passes=30):
    if loader is None:
        loader = val_loader

    fusion_model.eval()
    enc_text.eval(); enc_audio.eval(); enc_video.eval()

    preds_det, proba_det = [], []
    proba_mc_mean, proba_mc_std = [], []
    labels_all = []

    with torch.no_grad():
        for t, a, v, y in loader:
            t, a, v = t.to(DEVICE), a.to(DEVICE), v.to(DEVICE)

            # Deterministic prediction
            with autocast(enabled=USE_AMP):
                logits = fusion_model(t, a, v)

            proba = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            preds = torch.argmax(logits, dim=1).cpu().numpy()

            preds_det.extend(preds)
            proba_det.extend(proba)

            # MC-Dropout uncertainty
            mean_l, std_l = mc_predict(fusion_model, t, a, v, n_passes=mc_passes)

            proba_mc_mean.extend(torch.softmax(mean_l, dim=1)[:, 1].cpu().numpy())
            proba_mc_std.extend(std_l[:, 1].cpu().numpy())

            labels_all.extend(y.cpu().numpy())   # ✅ FIXED

    fusion_model.eval()  # restore eval mode after mc_predict

    print(f"\n{'='*65}")
    print(f"  {label} — Deterministic")
    print(f"{'='*65}")

    print(classification_report(labels_all, preds_det,
                                target_names=le.classes_, digits=4))

    cm = confusion_matrix(labels_all, preds_det)
    cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
    print("Confusion Matrix:\n", cm_df)

    try:
        auc_det = roc_auc_score(labels_all, proba_det)
        auc_mc  = roc_auc_score(labels_all, proba_mc_mean)
        print(f"\nROC-AUC (deterministic) : {auc_det:.4f}")
        print(f"ROC-AUC (MC-Dropout)    : {auc_mc:.4f}")
    except ValueError as e:
        print(f"\nAUC not defined: {e}")   # ✅ FIXED

    mean_unc = np.mean(proba_mc_std)
    print(f"Mean MC uncertainty (std on PTSD prob): {mean_unc:.4f}")
    print("  (lower = more confident; report this in clinical validation section)")

    return cm_df


print("✅ Evaluation helper ready (deterministic + MC-Dropout).")

✅ Evaluation helper ready (deterministic + MC-Dropout).


In [28]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 10 — Train all four fusion models
# ══════════════════════════════════════════════════════════════════════════════

print("\n🚀 Training Early Fusion…")
early_model, early_hist = train_model(
    EarlyFusionModel(), name="early_fusion", reset_encoders=True
)

print("\n🚀 Training Hybrid Fusion…")
hybrid_model, hybrid_hist = train_model(
    HybridFusionModel(), name="hybrid_fusion", reset_encoders=True
)

print("\n🚀 Training Late Fusion…")
late_model, late_hist = train_model(
    LateFusionModel(), name="late_fusion", reset_encoders=True
)

print("\n🚀 Training Cross-Modal Attention Fusion…")
attn_model, attn_hist = train_model(
    CrossModalAttentionFusion(), name="attn_fusion", lr=CFG["lr"] * 0.3, reset_encoders=True
)


🚀 Training Early Fusion…
[early_fusion] Ep 001/60 | TL 0.4536 | VL 0.3657 | Acc 0.9231 | F1 0.9231 | AUC 0.9894 | Joint 0.9551 | LR 1.00e-04
[early_fusion] Ep 002/60 | TL 0.1796 | VL 0.2674 | Acc 0.9744 | F1 0.9743 | AUC 0.9947 | Joint 0.9844 | LR 2.00e-04
[early_fusion] Ep 003/60 | TL 0.1476 | VL 0.2819 | Acc 0.9231 | F1 0.9231 | AUC 1.0000 | Joint 0.9600 | LR 3.00e-04
[early_fusion] Ep 004/60 | TL 0.1257 | VL 0.2242 | Acc 1.0000 | F1 1.0000 | AUC 1.0000 | Joint 1.0000 | LR 3.00e-04
[early_fusion] Ep 005/60 | TL 0.0788 | VL 0.2254 | Acc 1.0000 | F1 1.0000 | AUC 1.0000 | Joint 1.0000 | LR 2.99e-04
[early_fusion] Ep 006/60 | TL 0.1889 | VL 0.2314 | Acc 0.9744 | F1 0.9743 | AUC 1.0000 | Joint 0.9870 | LR 2.98e-04
[early_fusion] Ep 007/60 | TL 0.1475 | VL 0.2274 | Acc 0.9744 | F1 0.9743 | AUC 1.0000 | Joint 0.9870 | LR 2.96e-04
[early_fusion] Ep 008/60 | TL 0.0876 | VL 0.2174 | Acc 1.0000 | F1 1.0000 | AUC 1.0000 | Joint 1.0000 | LR 2.94e-04
[early_fusion] Ep 009/60 | TL 0.0910 | VL 0.21

In [29]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 11 — Evaluate all models
# ══════════════════════════════════════════════════════════════════════════════

evaluate_model(early_model,  label="Early Fusion")
evaluate_model(hybrid_model, label="Hybrid Fusion")
evaluate_model(late_model,   label="Late Fusion")
evaluate_model(attn_model,   label="Cross-Modal Attention Fusion")



  Early Fusion — Deterministic
              precision    recall  f1-score   support

     NO PTSD     0.8750    0.7778    0.8235        18
        PTSD     0.8261    0.9048    0.8636        21

    accuracy                         0.8462        39
   macro avg     0.8505    0.8413    0.8436        39
weighted avg     0.8487    0.8462    0.8451        39

Confusion Matrix:
          NO PTSD  PTSD
NO PTSD       14     4
PTSD           2    19

ROC-AUC (deterministic) : 0.9418
ROC-AUC (MC-Dropout)    : 0.9471
Mean MC uncertainty (std on PTSD prob): 0.0276
  (lower = more confident; report this in clinical validation section)

  Hybrid Fusion — Deterministic
              precision    recall  f1-score   support

     NO PTSD     0.6000    0.5000    0.5455        18
        PTSD     0.6250    0.7143    0.6667        21

    accuracy                         0.6154        39
   macro avg     0.6125    0.6071    0.6061        39
weighted avg     0.6135    0.6154    0.6107        39

Confusio

,NO PTSD,PTSD
NO PTSD,12,6
PTSD,12,9


In [31]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 12 — Training curves + summary table
# ══════════════════════════════════════════════════════════════════════════════

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

models_hist = {
    "Early":  early_hist,
    "Hybrid": hybrid_hist,
    "Late":   late_hist,
    "Attn":   attn_hist,
}

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
metrics   = [("train_loss", "Train Loss"), ("val_loss", "Val Loss"),
             ("f1",         "Val Macro-F1"), ("auc", "Val ROC-AUC")]

for ax, (col, title) in zip(axes, metrics):
    for name, h in models_hist.items():
        if col in h.columns:
            ax.plot(h["epoch"], h[col], label=name)
    ax.set_title(title); ax.set_xlabel("Epoch")
    ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
out_png = os.path.join(CFG["output_dir"], "training_curves.png")
plt.savefig(out_png, dpi=150)
plt.show()
print("✅ Curves saved →", out_png)

# Summary table
rows = []
for name, h in models_hist.items():
    best = h.loc[h["joint"].idxmax()]
    rows.append({"Model": name, "Best Epoch": int(best["epoch"]),
                 "Val Loss": f"{best['val_loss']:.4f}",
                 "Macro-F1": f"{best['f1']:.4f}",
                 "ROC-AUC":  f"{best['auc']:.4f}",
                 "Joint":    f"{best['joint']:.4f}"})
summary_df = pd.DataFrame(rows).set_index("Model")
print("\n── Model comparison ──")
print(summary_df.to_string())
summary_df.to_csv(os.path.join(CFG["output_dir"], "model_comparison.csv"))


✅ Curves saved → embeddings\training_curves.png

── Model comparison ──
        Best Epoch Val Loss Macro-F1 ROC-AUC   Joint
Model                                               
Early            4   0.2242   1.0000  1.0000  1.0000
Hybrid           3   0.2564   1.0000  1.0000  1.0000
Late             6   0.4047   1.0000  1.0000  1.0000
Attn             5   0.2358   1.0000  1.0000  1.0000


In [32]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 13 — Save all artefacts
# ══════════════════════════════════════════════════════════════════════════════

import pickle

np.save(os.path.join(CFG["output_dir"], "text_embeddings.npy"),  text_embeddings)
np.save(os.path.join(CFG["output_dir"], "audio_embeddings.npy"), audio_embeddings)
np.save(os.path.join(CFG["output_dir"], "video_embeddings.npy"), video_embeddings)

with open(os.path.join(CFG["output_dir"], "scalers.pkl"), "wb") as f:
    pickle.dump({"text": sc_text, "audio": sc_audio, "video": sc_video}, f)

with open(os.path.join(CFG["output_dir"], "label_encoder.pkl"), "wb") as f:
    pickle.dump(le, f)

for name, h in models_hist.items():
    h.to_csv(os.path.join(CFG["output_dir"], f"history_{name.lower()}.csv"), index=False)

print("✅ Artefacts saved to :", CFG["output_dir"])
print("   Models saved to   :", CFG["model_dir"])
print("\nFiles written:")
for f in sorted(os.listdir(CFG["output_dir"])):
    size = os.path.getsize(os.path.join(CFG["output_dir"], f))
    print(f"  {f:45s}  {size/1024:.1f} KB")


✅ Artefacts saved to : embeddings
   Models saved to   : saved_models

Files written:
  audio_embeddings.npy                           1365.1 KB
  history_attn.csv                               1.1 KB
  history_early.csv                              1.1 KB
  history_hybrid.csv                             1.2 KB
  history_late.csv                               1.2 KB
  label_encoder.pkl                              0.3 KB
  model_comparison.csv                           0.2 KB
  scalers.pkl                                    102.8 KB
  text_embeddings.npy                            585.1 KB
  training_curves.png                            186.1 KB
  video_embeddings.npy                           1365.1 KB


In [1]:
#Test :
!python ptsd_inference_pipeline.py "D:\Downloads\my-team-helped-me-stay-strong-and-stay-true-to-the-program-when-i-did-not-want_RdZ5TGF8.mp4" --transcript

PTSD Risk Prediction
Label           : NO PTSD
P(PTSD)         : 0.4703
95% CI          : [0.3875, 0.5530]
Uncertainty     : 0.0422
Confidence      : High
------------------------------------------------------------
Per-model predictions:
early_fusion             : P=0.4644  ->  NO PTSD
------------------------------------------------------------
Stage latency (seconds):
ingest                   : 0.479s
preprocess               : 5.847s
embed_text               : 0.259s
embed_audio              : 0.757s
embed_video              : 0.423s
fusion                   : 0.022s
mc_dropout               : 0.175s
TOTAL                    : 7.979s
TOTAL                    : 15.941s


00:23:28  [INFO    ]  ptsd — Scalers + label encoder loaded — classes: ['NO PTSD' 'PTSD']
00:23:29  [INFO    ]  ptsd — Loaded early_fusion  (GPU alloc=21MB  reserved=29MB)
00:23:30  [INFO    ]  ptsd — Tokenizer loaded: mental/mental-roberta-base
Some weights of RobertaModel were not initialized from the model checkpoint at mental/mental-roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
00:23:34  [INFO    ]  ptsd — Text model loaded: mental/mental-roberta-base
00:23:35  [INFO    ]  ptsd — EfficientNet-B4 loaded (audio)
00:23:36  [INFO    ]  ptsd — EfficientNet-B4 loaded (video)
00:23:36  [INFO    ]  ptsd — Pipeline ready on cuda  GPU alloc=660MB  reserved=673MB
00:23:36  [INFO    ]  ptsd — Ingest: duration=44.8s  target_frames=7
00:23:37  [INFO    ]  ptsd — Ingest done: audio=1.0s  frames=7
00:23:37  [WARNING ]  ptsd — Audio RM